In [2]:
# this file depends on many definitions found in configuration_eval
%run configuration_eval.ipynb

import json
import datetime

from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv("vars.env")

client = Anthropic()
SYSTEM_PROMPT = open("whisper_continuation_prompt.md").read()

#performs worse
#SYSTEM_PROMPT = open("whisper_terminology_prompt.md").read()

def run_prompt_engineering_eval(transcript_model, model, mp3_file, reference_srt):

    diagnostics_dir = os.path.join("../outputs/prompt_engineering_outputs", datetime.datetime.now().strftime("%H:%M:%S"))
    os.makedirs(diagnostics_dir, exist_ok=True)
    file_prefix = os.path.splitext(os.path.basename(mp3_file))[0][:5]

    srt1, transcription1 = whisper_transcribe(mp3_file, transcript_model, "", False)

    with open(os.path.join(diagnostics_dir, file_prefix + "transcription1.txt"), "w", encoding="utf-8") as f:
        f.write(transcription1)

    start_srt = os.path.join("../outputs/prompt_engineering_outputs", "srt1.srt")
    with open(start_srt, "w", encoding="utf-8") as f:
        f.write(srt1)

    resp = client.messages.create(
        model=model,
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": transcription1},
        ],
    )

    # skip thinking blocks
    whisper_prompt = next((block.text for block in resp.content if block.type == "text"), None)

    with open(os.path.join(diagnostics_dir, file_prefix + "whisper_prompt.txt"), "w", encoding="utf-8") as f:
        f.write(whisper_prompt)

    #finally the step where we get a transcription using our conditioning prompt generated at runtime!
    srt2, transcription2 = whisper_transcribe(mp3_file, transcript_model, whisper_prompt, False)

    with open(os.path.join(diagnostics_dir, file_prefix + "transcription2.txt"), "w", encoding="utf-8") as f:
        f.write(transcription2)

    end_srt = os.path.join("../outputs/prompt_engineering_outputs", "srt2.srt")
    with open(end_srt, "w", encoding="utf-8") as f:
        f.write(srt2)

    #calculate_wer from configuration_eval.ipynb
    score1 = calculate_wer(reference_srt, start_srt)
    score2 = calculate_wer(reference_srt, end_srt)
    with open(os.path.join(diagnostics_dir, file_prefix + "scores.txt"), "w", encoding="utf-8") as f:
        f.write(f"{score1}, {score2}")

    return score1, score2

def multi_prompt_engineering_eval(transcript_model, model, eval_path):
    scores1 = {}
    scores2 = {}

    for entry in sorted(os.listdir(eval_path)):
        folder = os.path.join(eval_path, entry)
        if not os.path.isdir(folder):
            continue

        reference_srt = os.path.join(folder, "corrected_transcript.srt")
        if not os.path.exists(reference_srt):
            continue

        mp3_files = glob.glob(os.path.join(folder, "*.mp3"))
        if not mp3_files:
            continue
        mp3_file = mp3_files[0]

        scores1[entry], scores2[entry] = run_prompt_engineering_eval(transcript_model, model, mp3_file, reference_srt)
        
    mean_errorscore1 = sum(scores1.values()) / len(scores1) if scores1 else 0.0
    mean_errorscore2 = sum(scores2.values()) / len(scores2) if scores2 else 0.0
    return scores1, mean_errorscore1, scores2, mean_errorscore2


In [7]:
transcript_model = "whisper-large-v3-turbo"
eval_path = "../golden_set"
model = "claude-sonnet-5"

scores1, score1, scores2, score2 = multi_prompt_engineering_eval(transcript_model, model, eval_path)

print(f"Average before score: {score1}")
print(f"Average after score: {score2}")
if score1 > score2:
    print("After auto prompt engineering, transcript accuracy improved.")
else:
    print("After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.")

Average before score: 0.02499939628287887
Average after score: 0.03506278353751247
After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.


In [8]:
transcript_model = "whisper-large-v3"
eval_path = "../golden_set"
model = "claude-sonnet-5"

scores1, score1, scores2, score2 = multi_prompt_engineering_eval(transcript_model, model, eval_path)

print(f"Average before score: {score1}")
print(f"Average after score: {score2}")
if score1 > score2:
    print("After auto prompt engineering, transcript accuracy improved.")
else:
    print("After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.")

Average before score: 0.025637735459090174
Average after score: 0.08953063528876715
After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.


In [3]:
transcript_model = "whisper-1"
eval_path = "../golden_set"
model = "claude-sonnet-5"

scores1, score1, scores2, score2 = multi_prompt_engineering_eval(transcript_model, model, eval_path)

print(f"Average before score: {score1}")
print(f"Average after score: {score2}")
if score1 > score2:
    print("After auto prompt engineering, transcript accuracy improved.")
else:
    print("After auto prompt engineering, transcript accuracy did not improve. Probably this model is a poor candidate for this workflow.")

# --- visualization: WER before/after, next to the workflow that produced it ---
from IPython.display import HTML, display

def render_prompt_engineering_report(score1, score2, transcript_model, model,
                                     out_path="../outputs/prompt_engineering_outputs/report.html"):
    improved = score2 < score1
    delta = (score1 - score2) / score1 * 100 if score1 else 0.0
    after_color = "#2f9e6e" if improved else "#c9503f"
    verdict = (f"WER down {delta:.1f}% after auto prompt engineering" if improved
               else f"WER up {abs(delta):.1f}% after auto prompt engineering")

    # bars scaled so the larger score fills the 160px plot area
    top = max(score1, score2) or 1.0
    h1, h2 = score1 / top * 160, score2 / top * 160

    steps = [
        (1, f"{transcript_model} transcription", "empty initial_prompt &rarr; baseline transcript", "#8e8e93"),
        (2, f"{model} prompt generation", "reads the baseline transcript, writes an initial_prompt", "#6b5bd6"),
        (3, f"{transcript_model} transcription", "same audio, conditioned on the generated prompt", after_color),
        (4, "WER vs. golden set", "both transcripts scored against the reference SRTs", "#1c1c1e"),
    ]

    rows = []
    for idx, (n, title, sub, color) in enumerate(steps):
        last = idx == len(steps) - 1
        rows.append(
            '<div style="display:flex;gap:12px;align-items:flex-start">'
            f'<div style="flex:0 0 24px;height:24px;border-radius:50%;background:{color};color:#fff;'
            'font-size:12px;font-weight:600;display:flex;align-items:center;justify-content:center">'
            f'{n}</div><div><div style="font-size:13px;font-weight:600">{title}</div>'
            f'<div style="font-size:12px;color:#6e6e73">{sub}</div></div></div>'
        )
        if not last:
            rows.append('<div style="margin:4px 0 4px 11px;width:2px;height:16px;background:#d1d1d6"></div>')
    workflow = "".join(rows)

    html = f"""
<div style="font-family:-apple-system,Segoe UI,Helvetica,Arial,sans-serif;color:#1c1c1e;background:#fff;
            border:1px solid #e5e5ea;border-radius:12px;padding:24px;max-width:920px">
  <div style="font-size:18px;font-weight:600">Auto Prompt Engineering &mdash; {transcript_model}</div>
  <div style="font-size:13px;color:#6e6e73;margin:2px 0 20px">
    prompt generated by {model} &middot; word error rate, lower is better</div>

  <div style="display:flex;gap:32px;flex-wrap:wrap">
    <div style="flex:0 0 250px">
      <div style="display:flex;align-items:flex-end;gap:36px;height:190px;
                  border-bottom:1px solid #d1d1d6;padding:0 12px">
        <div style="flex:1;text-align:center">
          <div style="font-size:13px;font-weight:600;margin-bottom:6px">{score1:.4f}</div>
          <div style="height:{h1:.1f}px;background:#8e8e93;border-radius:5px 5px 0 0"></div>
        </div>
        <div style="flex:1;text-align:center">
          <div style="font-size:13px;font-weight:600;margin-bottom:6px;color:{after_color}">{score2:.4f}</div>
          <div style="height:{h2:.1f}px;background:{after_color};border-radius:5px 5px 0 0"></div>
        </div>
      </div>
      <div style="display:flex;gap:36px;padding:8px 12px 0;font-size:12px;color:#6e6e73;text-align:center">
        <div style="flex:1">Before<br><span style="color:#aeaeb2">empty prompt</span></div>
        <div style="flex:1">After<br><span style="color:#aeaeb2">generated prompt</span></div>
      </div>
      <div style="margin-top:14px;font-size:13px;font-weight:600;color:{after_color}">{verdict}</div>
    </div>

    <div style="flex:1;min-width:290px;border-left:1px solid #e5e5ea;padding-left:32px">
      <div style="font-size:12px;font-weight:600;color:#6e6e73;margin-bottom:14px;
                  text-transform:uppercase;letter-spacing:.06em">Workflow</div>
      {workflow}
    </div>
  </div>
</div>
"""

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Chart written to {out_path}")
    return HTML(html)

render_prompt_engineering_report(score1, score2, transcript_model, model)


Average before score: 0.024695868667547676
Average after score: 0.0167478680378457
After auto prompt engineering, transcript accuracy improved.
Chart written to ../outputs/prompt_engineering_outputs/report.html
